# Devoir 5 - Solaire et applications
### Ève Castonguay & Tristan Tessier 

## **Question 3 - Dimensionnement d'un système PV et thermique**

In [1]:
# Imports
using Pkg
Pkg.activate(".")
Pkg.add("LaTeXStrings")   
Pkg.instantiate()
using DataFrames  
using Dates  
using CSV  
using Statistics  
using PlotlyJS 
using Unitful
using Printf
using LaTeXStrings 
using Glob # * dans les str

  Activating project at `~/Desktop/Solaire/D5`
     Project No packages added to or removed from `~/Desktop/Solaire/D5/Project.toml`
    Manifest No packages added to or removed from `~/Desktop/Solaire/D5/Manifest.toml`


In [ ]:
# Folder names
path_name = "/Users/evecastonguay/Library/CloudStorage/OneDrive-polymtl/Aut. 2026 - Polytechnique/Solaire/D5/"

# File names
integrals = "*-Integrals.csv"
checks = "*-Checks.csv"
thermal = "*-Thermal.csv"
pv = "*-PvDetails.csv"

file_list = [integrals,checks,thermal,pv]

/Users/evecastonguay/Library/CloudStorage/OneDrive-polymtl/Aut. 2026 - Polytechnique/Solaire/D5/depart

4-element Vector{String}:
 "*-Integrals.csv"
 "*-Checks.csv"
 "*-Thermal.csv"
 "*-PvDetails.csv"

In [ ]:
# Remove first December month
function remove_december(df)
    start_hour = 8760
    end_hour = 17520
    df_year = df[start_hour .<= df[:, 1] .<= end_hour, :]
    return df_year
end

## Simulation depart

In [ ]:
folder = "depart"
data_folder = path_name * folder
type_file = 1

df = CSV.read(glob(file_list[type_file],data_folder)[1], DataFrame) # glob(pattern,dossier) returns array

Row,TIME,slope,Ag,Vs,PPeakPv,CBat,QHeatLoad,QDhwLoad,QCoolLoad,PLightAppl,PCool,PAuxHeat,PAuxDhw,PPump,PTot,EincCol,QuCol,QSolTank,QTankHeat,QTankDhw,QTankEnv,dETank,etaCol,fSolThermal,EincPv,PPv,PLoad,PDumped,PGrid,dEBat,etaPv,fSolElectrical,Column33
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Missing
1,17520.0,45.0,10.0,20.0,5.0,10.0,2.76318e7,1.06599e7,1.36555e6,1.49595e7,5.46221e5,2.16159e7,2.56148e6,1.00238e6,4.06855e7,6.32222e7,2.23209e7,1.59794e7,6.0108e6,8.09846e6,1.87009e6,5.33984e-10,0.353055,0.368601,2.03388e8,3.0972e7,4.06862e7,1.05166e7,2.35568e7,-4.01219,0.15228,0.421002,missing


In [ ]:
folder = "depart"
data_folder = path_name * folder
type_file = 4 # PV

df = CSV.read(glob(file_list[type_file],data_folder)[1], DataFrame) # glob(pattern,dossier) returns array

# Mois de l'année : heure de fin de chaque mois (année non bissextile)
mois = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
heures_fin_mois = cumsum([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]) .* 24

# 3.1 - Get data from file
rename!(df, strip.(names(df))) # remove spaces after column names
stats = DataFrame(
    time = df.TIME .* u"hr",
    PPv = df.PPv .* u"kJ/hr",
    PGrid = df.PGrid .* u"kJ/hr",
    PTot = df.PTot .* u"kJ/hr",
    TPvCell = df.TPvCell .* u"°C"
)
# temps
Δt = stats.time .- vcat(stats.time[1], stats.time[1:end-1])
stats.Δt = Δt
# Fraction solaire electrique
Etot = stats.PTot .* Δt
Egrid = stats.PGrid .* Δt
Fe = (Etot.-Egrid) ./Etot
stats.Fe = Fe
stats.Egrid = Egrid 
# Fraction solaire mensuelle
h = ustrip.(u"hr", stats.time) .- 8760               # heure de l'année (0 → 8760)
m = searchsortedfirst.(Ref(heures_fin_mois), h)      # n° du mois de chaque pas de temps
Fe_mois = [(sum(Etot[m .== k]) - sum(Egrid[m .== k])) / sum(Etot[m .== k]) for k in 1:12]
# PGrid
Egrid_mois = [ustrip(u"GJ", sum(Egrid[m .== k])) for k in 1:12]   # énergie du réseau par mois [GJ]

## Section 4 - Graphique : énergie mensuelle tirée du réseau
trace_Egrid = bar(x = mois, y = Egrid_mois,
                  marker = attr(color = "#636EFA", opacity = 0.6,
                                line = attr(color = "black", width = 1)))
fig_Egrid = plot(trace_Egrid,
                 Layout(width = 800,
                    height = 400,
                    xaxis_title = "Mois",
                    yaxis_title = "Énergie du réseau [GJ]",
                    showlegend = false))
savefig(fig_Egrid, folder * "-energie_reseau_mensuelle.png", scale = 5)

"energie_reseau_mensuelle.png"